I/ Pipeline de préprocessing

In [ ]:
# --- Pipeline de préprocessing, réutilisable pour tous les modèles de la semaine ---
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Détection automatique des colonnes numériques vs catégorielles
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
print("Numériques :", num_cols)
print("Catégorielles :", cat_cols)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),        # normalisation, nécessaire pour la régression logistique
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),  # encodage des variables catégorielles
])

def make_model(clf):
    """Assemble le préprocessing et un classifieur dans un seul pipeline,
    pour garantir que le préprocessing est ré-appris uniquement sur les données d'entraînement
    à chaque validation croisée (évite toute fuite de données)."""
    return Pipeline([("prep", preprocess), ("clf", clf)])

In [ ]:
# --- Baseline : un modèle "bête" pour avoir un point de comparaison ---
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
print(f"Accuracy du dummy (prédit toujours 'reste') : {accuracy_score(y_test, dummy.predict(X_test)):.3f}")
# -> attendu ~0.735 : tout modèle sérieux doit largement dépasser ce chiffre

II/ Modèle de régression logistique

In [ ]:
# --- Modèle 1 : Régression logistique ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

RANDOM_STATE = 42

logreg = make_model(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Reste", "Churn"]).plot(cmap="Blues")

print(classification_report(y_test, y_pred, target_names=["Reste", "Churn"]))

In [ ]:
# --- Courbe ROC et AUC ---
from sklearn.metrics import roc_curve, roc_auc_score

y_proba = logreg.predict_proba(X_test)[:, 1]  # probabilité de churn
fpr, tpr, seuils = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, color=COULEUR_CHURN, linewidth=2, label=f"Rég. logistique (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Hasard (AUC = 0.5)")
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs (rappel)")
plt.title("Courbe ROC — régression logistique")
plt.legend()
plt.show()